# 03b - Topic modeling clásico: NMF y LDA

Esta notebook continúa el recorrido del Práctico 3. Después de probar KMeans con distintas representaciones, acá se exploran dos modelos clásicos de topic modeling: NMF y LDA.

La intención no es presentar estos modelos como solución final, sino entender qué aportan frente al clustering: en lugar de agrupar documentos en el espacio vectorial, buscan representar los textos como combinaciones de tópicos y describir cada tópico por sus palabras más importantes.

## Preparación del corpus

Se trabaja con `df_final.csv`, generado en el práctico anterior. Para mantener la reproducibilidad, los artefactos derivados se guardan en `data/processed/` y los resúmenes comparativos en `reports/`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

nltk.download("stopwords", quiet=True)

PROCESSED_DIR = Path("../data/processed") if Path.cwd().name == "notebooks" else Path("data/processed")
REPORTS_DIR = Path("../reports") if Path.cwd().name == "notebooks" else Path("reports")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [2]:
df = pd.read_csv(PROCESSED_DIR / "df_final.csv")
print(df.shape)
df[["rawContent_clean", "pysentimiento"]].head()

(8882, 78)


,rawContent_clean,pysentimiento
0,Boluda ahí dice que yo debería pesar 43kg eso ...,NEG
1,"Jugador de +30 años, vendehumo, con tendencias...",NEG
2,La tabla de mi pediatra cuando tenia 6 años y ...,NEG
3,chupame un huevo como voy a pesar 48 kg midien...,NEG
4,Eso me da curiosidad en la gente obesa. Miro k...,NEU


In [3]:
stop_words = set(stopwords.words("spanish"))
custom_stopwords = {
    "si", "no", "sos", "vos", "jaja", "ja", "q", "xq", "mas", "más", "solo", "la", "el",
    "ser", "tener", "hacer", "hace", "va", "re", "ahora", "bien", "mal", "puede", "pueden",
    "gente", "persona", "personas", "cosa", "cosas", "día", "dias", "días", "año", "años", "vez",
    "obesidad", "obeso", "obesa", "obesos", "obesas", "sobrepeso", "gordo", "gorda", "gordos", "gordas",
    "gordura", "mórbida", "morbida", "mórbido", "morbido",
    "vida", "siempre", "nunca", "dice", "decir", "dijo", "tan", "mejor", "nadie", "igual",
    "bueno", "buena", "tipo", "mismo", "misma", "voy", "anda", "mira", "entonces", "hoy",
    "veces", "ver", "vas", "después", "despues", "lado", "dan", "parece", "ganas", "tampoco",
    "menos", "ahí", "ahi", "hablar", "habla", "pasar", "falta", "creo", "cuenta", "dicen",
    "claro", "cada", "acá", "aca", "cómo", "como", "toda", "todo", "dos", "fin", "favor",
}
all_stopwords = stop_words.union(custom_stopwords)

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-záéíóúüñ0-9 ]+", " ", text)
    tokens = [token for token in text.split() if token not in all_stopwords and len(token) > 2]
    return " ".join(tokens)

df["topic_text"] = df["rawContent_clean"].apply(normalize_text)
docs = df["topic_text"].fillna("").tolist()

df[["rawContent_clean", "topic_text"]].head()


,rawContent_clean,topic_text
0,Boluda ahí dice que yo debería pesar 43kg eso ...,boluda debería pesar 43kg orto empezando cuida...
1,"Jugador de +30 años, vendehumo, con tendencias...",jugador vendehumo tendencias lesionarse tribun...
2,La tabla de mi pediatra cuando tenia 6 años y ...,tabla pediatra tenia decía
3,chupame un huevo como voy a pesar 48 kg midien...,chupame huevo pesar midiendo casi quién hizo l...
4,Eso me da curiosidad en la gente obesa. Miro k...,curiosidad miro kilos mortales veo placer toca...


## Representación documento-término

Para NMF se usa TF-IDF, porque el modelo trabaja bien con pesos no negativos que reducen el peso de términos demasiado frecuentes. Para LDA se usa una matriz de conteos, porque su formulación probabilística parte de frecuencias de palabras.

No se fuerzan embeddings Sentence-BERT en esta notebook: NMF y LDA son modelos de tópicos clásicos basados en términos. Los embeddings contextuales se retoman en la notebook de BERTopic.

In [4]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.9, min_df=10, ngram_range=(1, 2), max_features=5000)
count_vectorizer = CountVectorizer(max_df=0.9, min_df=10, ngram_range=(1, 2), max_features=5000)

X_tfidf = tfidf_vectorizer.fit_transform(docs)
X_count = count_vectorizer.fit_transform(docs)

print(f"Matriz TF-IDF para NMF: {X_tfidf.shape[0]} documentos x {X_tfidf.shape[1]} features")
print(f"Matriz de conteos para LDA: {X_count.shape[0]} documentos x {X_count.shape[1]} features")

Matriz TF-IDF para NMF: 8882 documentos x 1582 features
Matriz de conteos para LDA: 8882 documentos x 1582 features


## Funciones de evaluación

Se usan métricas simples pero comparables: diversidad de tópicos, NPMI aproximado sobre co-ocurrencias de términos, tamaño de tópicos y probabilidad media de asignación. En esta notebook se priorizan tablas sobre gráficos, porque las curvas no agregaban demasiado y hacían más pesada la lectura.


In [5]:
def top_terms_from_components(components, vectorizer, n_terms=12):
    feature_names = np.array(vectorizer.get_feature_names_out())
    rows = []
    for topic_id, component in enumerate(components):
        top_indices = component.argsort()[::-1][:n_terms]
        for rank, idx in enumerate(top_indices, start=1):
            rows.append({
                "topic": int(topic_id),
                "rank": rank,
                "term": feature_names[idx],
                "weight": float(component[idx]),
            })
    return pd.DataFrame(rows)

def topic_diversity(terms_df, top_n=10):
    top_terms = terms_df[terms_df["rank"] <= top_n]["term"].tolist()
    return len(set(top_terms)) / len(top_terms) if top_terms else np.nan

def mean_npmi(terms_df, binary_matrix, vectorizer, top_n=10):
    feature_index = {term: idx for idx, term in enumerate(vectorizer.get_feature_names_out())}
    binary_csc = binary_matrix.astype(bool).astype(int).tocsc()
    n_docs = binary_matrix.shape[0]
    scores = []
    for _, group in terms_df[terms_df["rank"] <= top_n].groupby("topic"):
        terms = [term for term in group.sort_values("rank")["term"] if term in feature_index]
        for i in range(len(terms)):
            for j in range(i + 1, len(terms)):
                idx_i = feature_index[terms[i]]
                idx_j = feature_index[terms[j]]
                col_i = binary_csc[:, idx_i]
                col_j = binary_csc[:, idx_j]
                p_i = col_i.sum() / n_docs
                p_j = col_j.sum() / n_docs
                p_ij = col_i.multiply(col_j).sum() / n_docs
                if p_ij > 0 and p_i > 0 and p_j > 0:
                    pmi = np.log(p_ij / (p_i * p_j))
                    scores.append(pmi / (-np.log(p_ij)))
    return float(np.mean(scores)) if scores else np.nan

def summarize_assignments(method, variant, doc_topic):
    topic = doc_topic.argmax(axis=1)
    probability = doc_topic.max(axis=1)
    sizes = pd.Series(topic).value_counts()
    return pd.DataFrame({
        "doc_id": df.index,
        "method": method,
        "variant": variant,
        "topic": topic,
        "probability": probability,
    }), {
        "method": method,
        "variant": variant,
        "n_topics": int(doc_topic.shape[1]),
        "min_topic_size": int(sizes.min()),
        "median_topic_size": float(sizes.median()),
        "max_topic_size": int(sizes.max()),
        "mean_assignment_probability": float(probability.mean()),
        "median_assignment_probability": float(np.median(probability)),
    }

def show_representative_docs(assignments_df, topic_id, n=5):
    selected = assignments_df[assignments_df["topic"] == topic_id].sort_values("probability", ascending=False).head(n)
    cols = ["rawContent_clean", "pysentimiento"]
    display(df.loc[selected["doc_id"], cols].assign(probability=selected["probability"].values))

## Selección de cantidad de tópicos

Se comparan varios valores de `k`. En NMF se mira el error de reconstrucción junto con diversidad/coherencia de términos. En LDA se mira perplexity/log-likelihood, también junto con diversidad/coherencia.

La selección no se toma como un óptimo automático. Para que la comparación con KMeans sea más directa, se conserva `k=12`: alcanza para mostrar una variedad razonable de tópicos sin volver ilegible la revisión cualitativa.


In [6]:
topic_range = [4, 6, 8, 10, 12, 15, 20, 25, 30]

nmf_sweep = []
for n_topics in topic_range:
    model = NMF(n_components=n_topics, init="nndsvda", random_state=RANDOM_STATE, max_iter=500)
    doc_topic = model.fit_transform(X_tfidf)
    terms_df = top_terms_from_components(model.components_, tfidf_vectorizer, n_terms=10)
    assignments_df, assignment_summary = summarize_assignments("NMF", f"nmf_tfidf_k{n_topics}", normalize(doc_topic, norm="l1"))
    nmf_sweep.append({
        **assignment_summary,
        "reconstruction_error": float(model.reconstruction_err_),
        "topic_diversity": topic_diversity(terms_df, top_n=10),
        "mean_npmi": mean_npmi(terms_df, X_count, count_vectorizer, top_n=10),
    })

nmf_sweep_df = pd.DataFrame(nmf_sweep)
nmf_sweep_df

,method,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,median_assignment_probability,reconstruction_error,topic_diversity,mean_npmi
0,NMF,nmf_tfidf_k4,4,386,660.5,7175,0.703766,0.748127,89.516672,0.975000,0.168198
1,NMF,nmf_tfidf_k6,6,354,555.0,6071,0.615253,0.615789,89.123559,0.883333,0.165778
2,NMF,nmf_tfidf_k8,8,310,820.0,3411,0.495607,0.439520,88.752599,0.850000,0.157356
3,NMF,nmf_tfidf_k10,10,299,605.5,3105,0.473932,0.404334,88.418361,0.790000,0.153501
4,NMF,nmf_tfidf_k12,12,156,295.0,4731,0.574107,0.568024,88.097228,0.816667,0.201577
5,NMF,nmf_tfidf_k15,15,154,279.0,3111,0.530845,0.501393,87.652852,0.820000,0.193389
6,NMF,nmf_tfidf_k20,20,133,387.5,1307,0.460504,0.399151,86.970119,0.790000,0.212964
7,NMF,nmf_tfidf_k25,25,119,251.0,1200,0.453591,0.393063,86.346184,0.796000,0.237699
8,NMF,nmf_tfidf_k30,30,106,253.5,781,0.440421,0.368761,85.736960,0.770000,0.236928


In [7]:
nmf_sweep_df[[
    "variant",
    "n_topics",
    "min_topic_size",
    "median_topic_size",
    "max_topic_size",
    "mean_assignment_probability",
    "reconstruction_error",
    "topic_diversity",
    "mean_npmi",
]].sort_values("mean_npmi", ascending=False)


,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,reconstruction_error,topic_diversity,mean_npmi
7,nmf_tfidf_k25,25,119,251.0,1200,0.453591,86.346184,0.796000,0.237699
8,nmf_tfidf_k30,30,106,253.5,781,0.440421,85.736960,0.770000,0.236928
6,nmf_tfidf_k20,20,133,387.5,1307,0.460504,86.970119,0.790000,0.212964
4,nmf_tfidf_k12,12,156,295.0,4731,0.574107,88.097228,0.816667,0.201577
5,nmf_tfidf_k15,15,154,279.0,3111,0.530845,87.652852,0.820000,0.193389
0,nmf_tfidf_k4,4,386,660.5,7175,0.703766,89.516672,0.975000,0.168198
1,nmf_tfidf_k6,6,354,555.0,6071,0.615253,89.123559,0.883333,0.165778
2,nmf_tfidf_k8,8,310,820.0,3411,0.495607,88.752599,0.850000,0.157356
3,nmf_tfidf_k10,10,299,605.5,3105,0.473932,88.418361,0.790000,0.153501


In [8]:
lda_sweep = []
for n_topics in topic_range:
    model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=20,
        n_jobs=-1,
    )
    doc_topic = model.fit_transform(X_count)
    terms_df = top_terms_from_components(model.components_, count_vectorizer, n_terms=10)
    assignments_df, assignment_summary = summarize_assignments("LDA", f"lda_count_k{n_topics}", doc_topic)
    lda_sweep.append({
        **assignment_summary,
        "perplexity": float(model.perplexity(X_count)),
        "log_likelihood": float(model.score(X_count)),
        "topic_diversity": topic_diversity(terms_df, top_n=10),
        "mean_npmi": mean_npmi(terms_df, X_count, count_vectorizer, top_n=10),
    })

lda_sweep_df = pd.DataFrame(lda_sweep)
lda_sweep_df

,method,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,median_assignment_probability,perplexity,log_likelihood,topic_diversity,mean_npmi
0,LDA,lda_count_k4,4,1876,2073.0,2860,0.625456,0.624625,1439.555118,-310481.856902,0.925000,0.076679
1,LDA,lda_count_k6,6,1316,1344.5,2074,0.589081,0.582898,1528.963665,-313054.488638,0.883333,0.076416
2,LDA,lda_count_k8,8,932,1063.5,1618,0.564852,0.562420,1597.524960,-314927.320552,0.937500,0.093975
3,LDA,lda_count_k10,10,748,833.0,1444,0.539664,0.549976,1620.019641,-315524.313715,0.900000,0.111776
4,LDA,lda_count_k12,12,621,681.0,1333,0.528326,0.541663,1692.939364,-317404.089756,0.900000,0.114274
5,LDA,lda_count_k15,15,445,540.0,1226,0.506174,0.516667,1757.255533,-318996.056045,0.920000,0.121748
6,LDA,lda_count_k20,20,304,399.0,1167,0.477010,0.506250,1864.295008,-321520.600035,0.925000,0.152789
7,LDA,lda_count_k25,25,260,331.0,1007,0.449444,0.453482,1955.094134,-323550.976674,0.928000,0.163350
8,LDA,lda_count_k30,30,207,272.0,964,0.429937,0.417103,2044.716625,-325464.603616,0.946667,0.171380


In [9]:
lda_sweep_df[[
    "variant",
    "n_topics",
    "min_topic_size",
    "median_topic_size",
    "max_topic_size",
    "mean_assignment_probability",
    "perplexity",
    "topic_diversity",
    "mean_npmi",
]].sort_values("mean_npmi", ascending=False)


,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,perplexity,topic_diversity,mean_npmi
8,lda_count_k30,30,207,272.0,964,0.429937,2044.716625,0.946667,0.171380
7,lda_count_k25,25,260,331.0,1007,0.449444,1955.094134,0.928000,0.163350
6,lda_count_k20,20,304,399.0,1167,0.477010,1864.295008,0.925000,0.152789
5,lda_count_k15,15,445,540.0,1226,0.506174,1757.255533,0.920000,0.121748
4,lda_count_k12,12,621,681.0,1333,0.528326,1692.939364,0.900000,0.114274
3,lda_count_k10,10,748,833.0,1444,0.539664,1620.019641,0.900000,0.111776
2,lda_count_k8,8,932,1063.5,1618,0.564852,1597.524960,0.937500,0.093975
0,lda_count_k4,4,1876,2073.0,2860,0.625456,1439.555118,0.925000,0.076679
1,lda_count_k6,6,1316,1344.5,2074,0.589081,1528.963665,0.883333,0.076416


Las métricas permiten comparar, pero no resuelven solas el problema. En topic modeling interesa que los tópicos sean legibles, que no repitan siempre las mismas palabras y que tengan una distribución razonable de documentos.

En las tablas se ve que aumentar la cantidad de tópicos puede mejorar alguna métrica puntual, pero también vuelve más fragmentada la lectura. Por eso se mantiene `k=12` como compromiso de comparabilidad e interpretabilidad, no como una verdad matemática del corpus.


In [10]:
NMF_TOPICS = 12
LDA_TOPICS = 12

nmf_model = NMF(n_components=NMF_TOPICS, init="nndsvda", random_state=RANDOM_STATE, max_iter=500)
nmf_doc_topic_raw = nmf_model.fit_transform(X_tfidf)
nmf_doc_topic = normalize(nmf_doc_topic_raw, norm="l1")
nmf_terms = top_terms_from_components(nmf_model.components_, tfidf_vectorizer, n_terms=12)
nmf_assignments, nmf_summary = summarize_assignments("NMF", f"nmf_tfidf_k{NMF_TOPICS}", nmf_doc_topic)
nmf_summary.update({
    "reconstruction_error": float(nmf_model.reconstruction_err_),
    "perplexity": np.nan,
    "log_likelihood": np.nan,
    "topic_diversity": topic_diversity(nmf_terms, top_n=10),
    "mean_npmi": mean_npmi(nmf_terms, X_count, count_vectorizer, top_n=10),
})

display(nmf_terms.pivot(index="topic", columns="rank", values="term"))

rank,1,2,3,4,5,6,7,8,9,10,11,12
topic,,,,,,,,,,,,
0,comer,feliz,debe,quiero,siento,dejar,semana,dejar comer,dulce,comer dulce,dormir,darle
1,mierda,puta,puto,asco,comen,hdp,cabeza,negro,pelotudo,panza,pelotas,pija
2,enfermedad,diabetes,mental,grave,crónica,además,riesgo,alguien,basta,elige,sino,anorexia
3,peso,bajar,bajar peso,cuestión,cuestión peso,mido,ideal,peso ideal,saludable,normal,bajo,nutricionista
4,dios,siento,mio,dios mio,puta,asi,asco,quiero,nivel,paso,momento,puedo
5,así,niños,etc,total,todas,asco,mil,feliz,necesito,pedazo,desagradable,aún
6,tenes,cara,podes,puta,adentro,foto,vieja,deja,nota,arterias,neuronas,vergüenza
7,problemas,salud,problemas salud,diabetes,enfermedades,riesgo,etc,mundial,trae,mental,gordofobia,usa
8,hambre,van,mucha,pueblo,todas,país,pobreza,congreso,pasan,niños,colesterol,masa


In [11]:
lda_model = LatentDirichletAllocation(
    n_components=LDA_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=20,
    n_jobs=-1,
)
lda_doc_topic = lda_model.fit_transform(X_count)
lda_terms = top_terms_from_components(lda_model.components_, count_vectorizer, n_terms=12)
lda_assignments, lda_summary = summarize_assignments("LDA", f"lda_count_k{LDA_TOPICS}", lda_doc_topic)
lda_summary.update({
    "reconstruction_error": np.nan,
    "perplexity": float(lda_model.perplexity(X_count)),
    "log_likelihood": float(lda_model.score(X_count)),
    "topic_diversity": topic_diversity(lda_terms, top_n=10),
    "mean_npmi": mean_npmi(lda_terms, X_count, count_vectorizer, top_n=10),
})

display(lda_terms.pivot(index="topic", columns="rank", values="term"))

rank,1,2,3,4,5,6,7,8,9,10,11,12
topic,,,,,,,,,,,,
0,enfermedad,tenes,alguien,foto,puedo,cirugía,nota,amor,así,grasa,gym,vieja
1,diabetes,riesgo,comer,peso,enfermedades,siento,ejercicio,física,dieta,factores,problemas,actividad
2,salud,problemas,problema,mundial,van,niños,peso,cuestión,millones,enfermedad,según,chicos
3,gato,mala,peor,mina,hijos,comer,boca,mierda,odio,feliz,deja,normal
4,peso,bajar,chica,pasa,bajar peso,ropa,paso,hacen,alguien,salud,poner,saben
5,mujer,mujeres,dios,etc,video,tratamiento,sociedad,hombres,fea,nuevo,realmente,negra
6,ustedes,argentina,señora,podes,poder,pelotudo,correr,pobre,cáncer,ley,parte,pedazo
7,cara,encima,asi,orto,madre,bastante,cantidad,dije,viejo,horrible,peso,concha
8,país,mundo,peso,razón,come,mayor,comer,cualquier,pacientes,amigo,argentina,importante


## Ejemplos representativos

Además de mirar palabras, se revisan tweets con alta probabilidad de pertenecer a algunos tópicos. Esto ayuda a detectar si las palabras representativas realmente se traducen en documentos coherentes.

In [12]:
for topic_id in [0, 1, 2]:
    print(f"NMF - tópico {topic_id}")
    show_representative_docs(nmf_assignments, topic_id, n=4)

NMF - tópico 0


,rawContent_clean,pysentimiento,probability
8753,"Mi papá siempre decía: ""si no fuera por la gor...",NEU,1.0
8696,El chelo no para de comer.esta rosando la obes...,NEG,1.0
6834,la del horóscopo me dijo básicamente q deje de...,NEG,1.0
7184,No hay que darle de comer más a este obeso,NEG,1.0


NMF - tópico 1


,rawContent_clean,pysentimiento,probability
8576,"Borja obeso de mierda volvete a Narcolombia, h...",NEG,1.0
971,y si son incogible obesa de mierda,NEG,1.0
932,Dicen que le tiró mierda y apoya al obeso tran...,NEG,1.0
2130,Cómo mierda está en el ejército con el sobrepe...,NEG,1.0


NMF - tópico 2


,rawContent_clean,pysentimiento,probability
1941,Tener sobrepeso es una enfermedad y no una ofensa,NEG,1.0
2016,"La obesidad , es una enfermedad. Muy lejos de ...",NEG,1.0
1676,Porque la obesidad es un a enfermedad y hay qu...,NEG,1.0
1311,La obesidad es una enfermedad,NEG,1.0


In [13]:
for topic_id in [0, 1, 2]:
    print(f"LDA - tópico {topic_id}")
    show_representative_docs(lda_assignments, topic_id, n=4)

LDA - tópico 0


,rawContent_clean,pysentimiento,probability
5668,El sistema de salud de fue reconocido en Méxic...,POS,0.974537
5482,"El Sistema de Salud de , presente en el “XVI C...",POS,0.961805
3845,Se necesita entrenar los músculos para ganar l...,NEG,0.949071
5530,El Centro de Obesidad de Malvinas Argentinas f...,POS,0.946078


LDA - tópico 1


,rawContent_clean,pysentimiento,probability
8123,¿En qué termina todo esto? En mayor incidencia...,NEG,0.968390
6142,No crítico tu físico! Marco en tu físico las c...,NEU,0.942706
4330,Resistencia a la insulina / SOP Diabetes Tipo ...,NEU,0.934524
7425,seamos sinceros y lo digo como una persona cuy...,NEG,0.934522


LDA - tópico 2


,rawContent_clean,pysentimiento,probability
4543,La salud mental y la obesidad son los grandes ...,NEG,0.946077
2614,Hay que tratarlas y evitarlas. Yo nunca me dro...,NEG,0.946077
1736,Es un día dedicado a abordar la epidemia mundi...,NEG,0.938886
1872,Más de 1.000 millones de personas sufren de ob...,NEG,0.934523


## Salidas para comparación

Se guardan tres salidas: métricas de barrido y selección, términos por tópico y asignaciones documento-tópico. Así la página final puede comparar KMeans, NMF, LDA y BERTopic con el mismo tipo de insumos.

In [14]:
classic_sweep_df = pd.concat([nmf_sweep_df, lda_sweep_df], ignore_index=True, sort=False)
classic_metrics_df = pd.DataFrame([nmf_summary, lda_summary])
classic_terms_df = pd.concat([
    nmf_terms.assign(method="NMF", variant=f"nmf_tfidf_k{NMF_TOPICS}"),
    lda_terms.assign(method="LDA", variant=f"lda_count_k{LDA_TOPICS}"),
], ignore_index=True)
classic_assignments_df = pd.concat([nmf_assignments, lda_assignments], ignore_index=True)

classic_sweep_df.to_csv(REPORTS_DIR / "classic_topic_model_sweep.csv", index=False)
classic_metrics_df.to_csv(REPORTS_DIR / "classic_topic_model_metrics.csv", index=False)
classic_terms_df.to_csv(REPORTS_DIR / "classic_topic_terms.csv", index=False)
classic_assignments_df.to_csv(PROCESSED_DIR / "classic_topic_document_assignments.csv", index=False)

display(classic_metrics_df)
display(classic_terms_df.head(20))
print("Guardado:")
print(REPORTS_DIR / "classic_topic_model_sweep.csv")
print(REPORTS_DIR / "classic_topic_model_metrics.csv")
print(REPORTS_DIR / "classic_topic_terms.csv")
print(PROCESSED_DIR / "classic_topic_document_assignments.csv")

,method,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,median_assignment_probability,reconstruction_error,perplexity,log_likelihood,topic_diversity,mean_npmi
0,NMF,nmf_tfidf_k12,12,156,295.0,4731,0.574107,0.568024,88.097228,NaN,NaN,0.816667,0.201577
1,LDA,lda_count_k12,12,621,681.0,1333,0.528326,0.541663,NaN,1692.939364,-317404.089756,0.900000,0.114274


,topic,rank,term,weight,method,variant
0,0,1,comer,5.173238,NMF,nmf_tfidf_k12
1,0,2,feliz,0.479784,NMF,nmf_tfidf_k12
2,0,3,debe,0.478039,NMF,nmf_tfidf_k12
3,0,4,quiero,0.432457,NMF,nmf_tfidf_k12
4,0,5,siento,0.314632,NMF,nmf_tfidf_k12
5,0,6,dejar,0.225768,NMF,nmf_tfidf_k12
6,0,7,semana,0.211847,NMF,nmf_tfidf_k12
7,0,8,dejar comer,0.207968,NMF,nmf_tfidf_k12
8,0,9,dulce,0.199598,NMF,nmf_tfidf_k12
9,0,10,comer dulce,0.190755,NMF,nmf_tfidf_k12


Guardado:
../reports/classic_topic_model_sweep.csv
../reports/classic_topic_model_metrics.csv
../reports/classic_topic_terms.csv
../data/processed/classic_topic_document_assignments.csv


## Cierre

NMF y LDA aportan una capa interpretativa que KMeans no trae de forma directa: cada tópico queda definido por palabras y cada documento puede entenderse como mezcla de tópicos. En esta versión se dejaron solo tablas y ejemplos, porque los gráficos de términos no agregaban información suficiente.

Aun con stopwords ampliadas, aparecen tópicos amplios y algunas repeticiones, algo esperable en un corpus de tweets corto y ruidoso. Esta limitación justifica pasar a BERTopic, que combina embeddings contextuales con clustering y representación por términos.
